<a href="https://colab.research.google.com/github/kramos213/LSantimateo/blob/main/Proyecto_Konzerta_KR.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Análisis de ofertas de empleo de Konzerta**

**Curso:** Introducción a la Programación para Ciencia de Datos

**Integrantes:** Francia Vergara, Luis Santimateo y Yenia Rojas

**Fuente:** Konzerta

**Archivo utilizado:** Konzerta 08-oct-2026.xlsx

**Fecha de obtención del archivo:** 16-oct-2026



###**Contexto del proyecto**

En este proyecto buscamos conocer las características de las oportunidades de empleo incluidas en el archivo de Konzerta. Nos interesa identificar qué áreas profesionales concentran más anuncios y vacantes, dónde se ubican las oportunidades y qué modalidades de trabajo ofrecen.

También exploraremos la información salarial, su relación con la experiencia solicitada y la cantidad de anuncios publicados por semana. Para interpretar estos resultados, tendremos en cuenta los datos faltantes y los valores que requieren revisión.

###**Objetivo del análisis**

Analizar las ofertas de empleo de Konzerta mediante la revisión, limpieza y exploración de los datos, para responder las preguntas de nuestra propuesta y comunicar los hallazgos con tablas y gráficos.

El alcance del análisis corresponde a los anuncios incluidos en el archivo utilizado.

###**Preguntas de análisis**

¿Qué áreas y subáreas profesionales presentan mayor demanda según las
ofertas y vacantes publicadas?

¿Cómo varían los salarios ofrecidos según el nivel de experiencia requerido en las ofertas que cuentan con información salarial?

¿Qué modalidades de trabajo son más frecuentes y cómo cambian según el área?

¿Cómo se distribuyen las oportunidades laborales según área profesional, nivel laboral y ubicación geográfica?

¿Cómo evoluciona el volumen de ofertas publicadas por semana entre julio y octubre de 2026?

In [ ]:
try:
    # Importamos files, que permite trabajar con la carga y descarga de archivos.
    from google.colab import files
    # Permite cargar el dataset desde la computadora.
    archivos = files.upload()
except ModuleNotFoundError:
    print("No se detectó el entorno. Se inicializa 'archivos' vacío.")
    archivos = {}
except Exception as e:
    print(f"Ocurrió un error al cargar los archivos: {e}")
    archivos = {}

Saving Konzerta 08-oct-2026.xlsx to Konzerta 08-oct-2026 (1).xlsx


# **Lectura y revisión inicial de los datos**

Leemos el archivo de Konzerta y conservamos una tabla original como referencia. Después creamos una copia para trabajar y comparar los cambios realizados durante la limpieza.

Cada fila representa un anuncio de empleo. Revisamos el tamaño del archivo, sus primeros registros y los tipos de datos para conocer su estructura antes de comenzar la limpieza.

In [ ]:
import pandas as pd
from io import BytesIO

# Leemos directamente el Excel que acabamos de subir
nombre_archivo = list(archivos.keys())[0]
crudos = pd.read_excel(BytesIO(archivos[nombre_archivo]))

# Trabajamos sobre una copia de los datos originales
konzerta = crudos.copy()

# Revisamos el tamaño del archivo
print (f"El archivo contiene {crudos.shape[0]:,} anuncios " f"y {crudos.shape[1]} columnas.")

# Mostramos una muestra con las columnas principales
display(konzerta[["id", "titulo", "area", "provincia", "modalidad"]].head(3))

# Revisamos los tipos de datos y los valores presentes
konzerta.info()

El archivo contiene 3,011 anuncios y 31 columnas.


,id,titulo,area,provincia,modalidad
0,1118469088,Ortodoncista,"Salud, Medicina y Farmacia",Panamá,Presencial
1,1118473088,Gerente de Auditoría Externa,"Administración, Contabilidad y Finanzas",Panamá,Presencial
2,1118473021,Ejecutivo de Atencion al Cliente - Canales Dig...,"Atención al Cliente, Call Center y Telemarketing",Panamá,Presencial


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3011 entries, 0 to 3010
Data columns (total 31 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   id                    3011 non-null   int64         
 1   titulo                3011 non-null   object        
 2   empresa               1846 non-null   object        
 3   empresa_confidencial  3011 non-null   bool          
 4   industria             1788 non-null   object        
 5   empresa_tamano        1820 non-null   object        
 6   fecha_publicacion     3011 non-null   datetime64[ns]
 7   fecha_finalizacion    3011 non-null   datetime64[ns]
 8   provincia             3011 non-null   object        
 9   localidad             2969 non-null   object        
 10  area                  3011 non-null   object        
 11  subarea               3011 non-null   object        
 12  nivel_laboral         2758 non-null   object        
 13  tipo_trabajo      

## **Revisión de valores faltantes**

Revisamos cuántos datos faltan en cada columna y qué porcentaje representan sobre el total de anuncios. Esto nos ayuda a reconocer las limitaciones de la información y a decidir cómo tratarla durante la limpieza.

Las decisiones sobre estos faltantes se documentan en la bitácora para explicar qué hicimos y por qué.

In [ ]:
# Medimos los faltantes sobre la tabla original, antes de la limpieza
total_anuncios = len(crudos)
faltantes_por_columnas = crudos.isna().sum()

porcentaje_por_columna = (faltantes_por_columnas / total_anuncios * 100).round(2)

# Reunimos cantidades y porcentajes en una sola tabla
resumen_faltante = pd.DataFrame({"cantidad_faltante": faltantes_por_columnas, "porcentaje_faltante": porcentaje_por_columna}).sort_values(by="cantidad_faltante", ascending=False)
display(resumen_faltante)

,cantidad_faltante,porcentaje_faltante
idiomas,2606,86.55
salario_min,2546,84.56
salario_tipo,2526,83.89
salario_max,2526,83.89
salario_frecuencia,2526,83.89
edad_min,2344,77.85
edad_max,2344,77.85
experiencia_meses,2159,71.70
educacion,1789,59.42
educacion_estado,1789,59.42


# **Revisión de duplicados**

Comprobamos si hay filas completamente iguales y si algún identificador de anuncio aparece más de una vez. Estas repeticiones podrían alterar los conteos de ofertas y vacantes.

Revisamos ambas situaciones porque un anuncio puede conservar el mismo identificador y tener diferencias en otros campos. Las decisiones sobre su tratamiento se documentan en la bitácora de limpieza.

In [ ]:
# Buscamos repeticiones en los datos originales
duplicados_completos = crudos.duplicated().sum()
duplicados_id = crudos["id"].duplicated().sum()

print("Filas completamente duplicadas:", duplicados_completos)
print("Repeticiones de id:", duplicados_id)

Filas completamente duplicadas: 0
Repeticiones de id: 0


**No se encontraron filas completamente duplicadas ni identificadores repetidos en los datos originales. Por este motivo, no es necesario eliminar anuncios por duplicación.**

# **Revisión de valores numéricos**

Revisamos las vacantes, los salarios, las edades y la experiencia para conocer sus valores habituales e identificar cifras que necesitan validación.

Primero consultamos las estadísticas de cada variable, como el promedio, la mediana y los valores mínimos y máximos. Después observamos los cinco anuncios con mayores salarios máximos registrados.

Esta revisión sirve para orientar la limpieza. Un valor extremo requiere evaluación antes de decidir su tratamiento. Además, la moneda de los importes salariales sigue pendiente de confirmar, lo que limita su comparación entre anuncios.

In [ ]:
# Seleccionamos las variables numéricas relevantes para el análisis
columnas_numericas = ["vacantes", "salario_min", "salario_max", "edad_min", "edad_max", "experiencia_meses"]
resumen_numerico = crudos[columnas_numericas].describe().T

# Redondeamos la presentación para facilitar la lectura
display(resumen_numerico.round(2))

# Revisamos los anuncios con mayores importes salariales registrados
columnas_salario = ["id", "titulo", "salario_min", "salario_max", "salario_frecuencia", "salario_tipo"]

display(crudos[columnas_salario].sort_values(by="salario_max", ascending=False).head(5))

,count,mean,std,min,25%,50%,75%,max
vacantes,3011.0,2.71,6.79,1.0,1.0,1.0,1.0,50.0
salario_min,465.0,2860.38,41710.19,1.0,630.0,750.0,1000.0,900000.0
salario_max,485.0,4278.46,61394.19,0.0,700.0,980.0,1450.0,1350000.0
edad_min,667.0,24.79,4.61,18.0,20.0,25.0,27.0,45.0
edad_max,667.0,42.14,7.34,23.0,36.0,43.0,45.0,99.0
experiencia_meses,852.0,28.32,18.46,12.0,12.0,24.0,36.0,120.0


,id,titulo,salario_min,salario_max,salario_frecuencia,salario_tipo
2081,1118419642,Desarrollador Junior Full Stack (Móvil y Web c...,900000.0,1350000.0,por Mes,BRUTO
1259,1118440234,Agente Comercial (Panama Metro),NaN,72176.0,por Mes,BRUTO
1751,1118428715,ASESOR DE VENTAS,NaN,62816.0,por Mes,BRUTO
739,1118452388,Director(a) de Logística Integral – México y P...,19900.0,20000.0,por Mes,BRUTO
1923,1118423515,Asistente Contable y Administrativo,12000.0,12001.0,por Mes,BRUTO


# **Limpieza de datos**

Aplicamos los ajustes identificados durante la revisión inicial. Cada ejecución comienza con una nueva copia de los datos originales para que podamos repetir el proceso y comparar los cambios.

Primero quitamos los espacios sobrantes en los nombres de empresa y convertimos la fecha de extracción a un formato de fecha y hora. Guardamos el perfil inicial de los datos y registramos cada decisión en la bitácora de limpieza.

In [ ]:
# Empezamos cada limpieza desde una copia de los datos originales
konzerta = crudos.copy()

# Guardamos los tipos y los faltantes para compararlos al finalizar
perfil_antes = pd.DataFrame({
    "tipo": konzerta.dtypes.astype(str),
    "faltantes": konzerta.isna().sum()
})

# Contamos los nombres con espacios sobrantes antes de corregirlos
empresas_con_espacios = (
    konzerta["empresa"].notna()
    & (konzerta["empresa"] != konzerta["empresa"].str.strip())
).sum()

konzerta["empresa"] = konzerta["empresa"].str.strip()

konzerta["extraido"] = pd.to_datetime(
    konzerta["extraido"], errors="raise"
)

# Iniciamos la bitácora con los cambios realizados y sus motivos
bitacora = pd.DataFrame({
    "Columna": ["empresa", "extraido"],
    "Justificación": [
        "Se eliminaron espacios al inicio y al final de "
        f"{empresas_con_espacios} nombres de empresa para unificar su formato. "
        "Los valores faltantes se conservaron.",
        "Se convirtió extraido a fecha y hora para poder ordenar "
        "y validar los registros por su fecha de extracción."
    ]
})

# **Tratamiento de faltantes y revisión de salarios**

En las categorías que utilizaremos para agrupar los anuncios, representamos la información ausente con la etiqueta “Sin dato”. Los faltantes numéricos se conservan como valores ausentes.

Para los salarios, aplicamos reglas que permiten identificar importes que requieren revisión. También utilizamos el rango intercuartílico (IQR), que mide la dispersión de la mitad central de los salarios máximos registrados.

**Clasificamos los anuncios en tres grupos:**

**Sin dato:** no tienen salario mínimo ni máximo registrado.

**Registro sin alerta:** tienen algún importe registrado y no activan las reglas aplicadas.

**Revisar:** presentan algún importe o rango que necesita validación.

Los importes se conservan. Estas categorías reflejan las comprobaciones realizadas; la moneda sigue pendiente de confirmar.

In [ ]:
# Etiquetamos las ausencias en las categorías que analizaremos
categorias = ["area", "subarea", "provincia", "localidad", "nivel_laboral", "modalidad"]

for columna in categorias:
    konzerta[columna] = (konzerta[columna].astype("string").str.strip().replace("", pd.NA).fillna("Sin dato"))

# Comprobamos los duplicados después de estandarizar las categorías
duplicados_eliminados = int(konzerta.duplicated().sum())
konzerta = konzerta.drop_duplicates().copy()

# Calculamos límites de referencia para los salarios máximos
q1 = konzerta["salario_max"].quantile(0.25)
q3 = konzerta["salario_max"].quantile(0.75)
iqr = q3 - q1

limite_inferior = q1 - 1.5 * iqr
limite_superior = q3 + 1.5 * iqr

# Señalamos importes y rangos que requieren revisión
alerta_salario = ((konzerta["salario_min"] <= 1) | (konzerta["salario_max"] <= 0) | (konzerta["salario_min"] > konzerta["salario_max"])
    | (konzerta["salario_max"] < limite_inferior) | (konzerta["salario_max"] > limite_superior))

con_salario = (konzerta["salario_min"].notna() | konzerta["salario_max"].notna())

konzerta["estado_salario"] = "Sin dato"
konzerta.loc[con_salario, "estado_salario"] = "Registro sin alerta"
konzerta.loc[con_salario & alerta_salario, "estado_salario"] = "Revisar"

# Contamos los anuncios de cada clasificación
display(konzerta["estado_salario"].value_counts())

,count
estado_salario,
Sin dato,2526
Registro sin alerta,414
Revisar,71


In [ ]:
# Documentamos las decisiones sobre los faltantes
nuevas_filas = []

for columna in crudos.columns:
    cantidad = int(crudos[columna].isna().sum())

    if cantidad > 0:
        if columna in categorias:
            motivo = (f"Se etiquetaron {cantidad} faltantes como Sin dato " "para conservar los anuncios en las agrupaciones.")
        else:
            motivo = (f"Se conservaron {cantidad} valores ausentes porque ""no hay evidencia para completarlos. Los cálculos ""usarán los datos disponibles.")

        nuevas_filas.append({"Columna": columna, "Justificación": motivo})

# Documentamos duplicados y alertas salariales
nuevas_filas.extend([{"Columna": "Todas las columnas", "Justificación": (f"Se eliminaron {duplicados_eliminados} filas completamente "
            "repetidas para evitar conteos dobles.")}, {"Columna": "estado_salario", "Justificación": ("Se conservaron los importes y se marcaron para revisión "
            "mínimos <= 1, máximos <= 0, rangos invertidos y máximos " f"fuera de [{limite_inferior:.2f}, {limite_superior:.2f}], "
            "calculado con 1.5 IQR. La moneda sigue sin verificar.")}])

# Añadimos las decisiones conservando las anotaciones anteriores
bitacora = (pd.concat([bitacora, pd.DataFrame(nuevas_filas)], ignore_index=True).drop_duplicates().reset_index(drop=True))

# Diferenciamos los nulos de las etiquetas Sin dato
perfil_despues = pd.DataFrame({"tipo": konzerta.dtypes.astype(str),"faltantes": konzerta.isna().sum(),"sin_dato": konzerta.eq("Sin dato").sum()})


# **Verificación de la limpieza**

Comparamos la cantidad de filas y columnas, los tipos de datos y los valores faltantes antes y después de la limpieza.

La columna sin_dato cuenta las etiquetas que representan información ausente. Esto permite distinguirlas de los valores nulos que conservamos.

Finalmente, presentamos la bitácora completa para explicar qué cambios realizamos y cuál fue su justificación.

In [ ]:
# Comparamos la cantidad de filas y columnas
comparacion_dimensiones = pd.DataFrame({"filas": [crudos.shape[0], konzerta.shape[0]],"columnas": [crudos.shape[1], konzerta.shape[1]]
}, index=["Antes", "Después"])

display(comparacion_dimensiones)

# Unimos los perfiles para comparar tipos y valores faltantes
comparacion_perfiles = perfil_antes.join(perfil_despues,how="outer",lsuffix="_antes",rsuffix="_despues")

display(comparacion_perfiles)

# Mostramos la bitácora completa con el texto alineado a la izquierda
display(bitacora.style.set_properties(**{"text-align": "left"}).set_table_styles([{"selector": "th", "props": [("text-align", "left")]}]))

,filas,columnas
Antes,3011,31
Después,3011,32


,tipo_antes,faltantes_antes,tipo_despues,faltantes_despues,sin_dato
apto_discapacidad,bool,0.0,bool,0,0
area,object,0.0,string,0,0
contratacion,object,598.0,object,598,0
descripcion,object,0.0,object,0,0
edad_max,float64,2344.0,float64,2344,0
edad_min,float64,2344.0,float64,2344,0
educacion,object,1789.0,object,1789,0
educacion_estado,object,1789.0,object,1789,0
empresa,object,1165.0,object,1165,0
empresa_confidencial,bool,0.0,bool,0,0


,Columna,Justificación
0,empresa,Se eliminaron espacios al inicio y al final de 115 nombres de empresa para unificar su formato. Los valores faltantes se conservaron.
1,extraido,Se convirtió extraido a fecha y hora para poder ordenar y validar los registros por su fecha de extracción.
2,empresa,Se conservaron 1165 valores ausentes porque no hay evidencia para completarlos. Los cálculos usarán los datos disponibles.
3,industria,Se conservaron 1223 valores ausentes porque no hay evidencia para completarlos. Los cálculos usarán los datos disponibles.
4,empresa_tamano,Se conservaron 1191 valores ausentes porque no hay evidencia para completarlos. Los cálculos usarán los datos disponibles.
5,localidad,Se etiquetaron 42 faltantes como Sin dato para conservar los anuncios en las agrupaciones.
6,nivel_laboral,Se etiquetaron 253 faltantes como Sin dato para conservar los anuncios en las agrupaciones.
7,contratacion,Se conservaron 598 valores ausentes porque no hay evidencia para completarlos. Los cálculos usarán los datos disponibles.
8,salario_min,Se conservaron 2546 valores ausentes porque no hay evidencia para completarlos. Los cálculos usarán los datos disponibles.
9,salario_max,Se conservaron 2526 valores ausentes porque no hay evidencia para completarlos. Los cálculos usarán los datos disponibles.


El proceso estandariza los espacios en los nombres de empresa, convierte la fecha de extracción y añade una clasificación de la información salarial. En las categorías seleccionadas, la etiqueta “Sin dato” permite incluir los anuncios con información incompleta. Los faltantes numéricos se conservan como valores ausentes.

En estado_salario, los campos del perfil anterior aparecen como NaN porque esa columna se creó durante la limpieza.

Los importes marcados como “Revisar” mantienen sus valores originales. Su clasificación indica que requieren validación antes de interpretar las diferencias salariales

**Fuente:** [Konzerta](https://www.konzerta.com/)

# **Diccionario de datos**

Este diccionario explica las columnas que utilizaremos en el análisis. Los tipos de datos corresponden a la tabla después de la limpieza.

Incluimos las 31 columnas originales y estado_salario, que creamos para distinguir la disponibilidad de importes salariales y los casos que requieren revisión.

In [ ]:
# Explicamos el significado de cada columna
descripciones = {"id": "Identificador del anuncio de empleo.",
    "titulo": "Nombre del puesto publicado.",
    "empresa": "Nombre de la empresa anunciante, cuando se informa.",
    "empresa_confidencial": "Indica si la empresa se presenta como confidencial.",
    "industria": "Sector de actividad registrado para la empresa.",
    "empresa_tamano": "Categoría de tamaño de la empresa.",
    "fecha_publicacion": "Fecha de publicación del anuncio.",
    "fecha_finalizacion": "Fecha de finalización indicada para el anuncio.",
    "provincia": "Provincia registrada para la oferta.",
    "localidad": "Localidad registrada para la oferta.",
    "area": "Área profesional a la que pertenece el anuncio.",
    "subarea": "Subárea o especialidad profesional del anuncio.",
    "nivel_laboral": "Nivel o jerarquía del puesto.",
    "tipo_trabajo": "Clasificación del trabajo, como Full-time o Part-time.",
    "modalidad": "Forma de trabajo: presencial, híbrida o remota.",
    "contratacion": "Tipo de contratación, como indeterminado o temporal.",
    "vacantes": "Cantidad de puestos ofrecidos en el anuncio.",
    "salario_min": "Importe salarial mínimo registrado; moneda sin confirmar.",
    "salario_max": "Importe salarial máximo registrado; moneda sin confirmar.",
    "salario_frecuencia": "Periodicidad del salario registrado, por ejemplo mensual.",
    "salario_tipo": "Tipo de importe salarial registrado, por ejemplo bruto.",
    "edad_min": "Edad mínima solicitada, en años.",
    "edad_max": "Edad máxima solicitada, en años.",
    "experiencia_meses": "Experiencia requerida registrada en meses.",
    "educacion": "Nivel educativo solicitado.",
    "educacion_estado": "Situación de los estudios, como graduado o en curso.",
    "idiomas": "Idiomas y niveles de dominio indicados en el anuncio.",
    "apto_discapacidad": "Indica si el anuncio señala que es apto para personas con discapacidad.",
    "descripcion": "Texto del anuncio con información del puesto y sus requisitos.",
    "url": "Enlace al anuncio original.",
    "extraido": "Fecha y hora de extracción registradas en el dataset.",
    "estado_salario": "Clasificación creada en la limpieza según disponibilidad de importes y alertas."}

# Obtenemos los tipos directamente de la tabla limpia
diccionario_datos = pd.DataFrame({"columna": konzerta.columns,"tipo": list(konzerta.dtypes.astype(str))})

diccionario_datos["descripcion"] = (diccionario_datos["columna"].map(descripciones))

display(diccionario_datos.style.set_properties(**{"text-align": "left"}).set_table_styles([
        {"selector": "th", "props": [("text-align", "left")]}]))

,columna,tipo,descripcion
0,id,int64,Identificador del anuncio de empleo.
1,titulo,object,Nombre del puesto publicado.
2,empresa,object,"Nombre de la empresa anunciante, cuando se informa."
3,empresa_confidencial,bool,Indica si la empresa se presenta como confidencial.
4,industria,object,Sector de actividad registrado para la empresa.
5,empresa_tamano,object,Categoría de tamaño de la empresa.
6,fecha_publicacion,datetime64[ns],Fecha de publicación del anuncio.
7,fecha_finalizacion,datetime64[ns],Fecha de finalización indicada para el anuncio.
8,provincia,string,Provincia registrada para la oferta.
9,localidad,string,Localidad registrada para la oferta.


# **Análisis exploratorio de datos**

###**Pregunta 1. Áreas y subáreas con mayor demanda**

¿Qué áreas y subáreas profesionales presentan mayor demanda según las ofertas y vacantes publicadas?

Comenzamos agrupando los anuncios por área profesional. Contaremos las ofertas y sumaremos las vacantes registradas.

Una oferta corresponde a un anuncio; las vacantes representan los puestos que ese anuncio ofrece. Esta diferencia permite observar tanto la cantidad de publicaciones como los puestos anunciados.

In [ ]:
# Contamos los anuncios y sumamos sus vacantes por área profesional
tabla_areas = (konzerta.groupby("area", dropna=False).agg(ofertas=("id", "size"),vacantes=("vacantes", "sum")).reset_index().sort_values(by="ofertas", ascending=False))

# Calculamos la participación de cada área en el total de anuncios
tabla_areas["porcentaje_ofertas"] = (tabla_areas["ofertas"] / len(konzerta) * 100).round(2)

display(tabla_areas)

,area,ofertas,vacantes,porcentaje_ofertas
4,"Comercial, Ventas y Negocios",801,2808,26.60
1,"Administración, Contabilidad y Finanzas",446,625,14.81
0,Abastecimiento y Logística,294,626,9.76
17,Oficios y Otros,225,617,7.47
24,"Tecnología, Sistemas y Telecomunicaciones",148,318,4.92
3,"Atención al Cliente, Call Center y Telemarketing",146,1608,4.85
9,Gastronomía y Turismo,143,314,4.75
18,Producción y Manufactura,142,268,4.72
12,Ingenierías,118,166,3.92
11,Ingeniería Civil y Construcción,99,166,3.29


In [ ]:
import plotly.express as px

# Seleccionamos las diez áreas con más anuncios
top_areas = tabla_areas.head(10).sort_values(by="ofertas")

fig = px.bar(top_areas, x="ofertas", y="area", orientation="h", text="ofertas", title="Las 10 áreas con más ofertas en el dataset de Konzerta",
    labels={"ofertas": "Número de ofertas (anuncios)" ,"area": "Área profesional"})

fig.update_layout(height=650)
fig.show()

Comercial, Ventas y Negocios encabezan las publicaciones del dataset con 801 anuncios, equivalentes al 26.60 % del total. Aunque Administración tiene más anuncios (446) que Atención al Cliente (146), esta última reúne más vacantes (1,608 frente a 625), lo que refleja una mayor cantidad de puestos por anuncio en promedio.

### **Análisis de subáreas profesionales con mayor demanda**

Dentro de las áreas identificadas, existen subáreas o especialidades específicas. Vamos a agrupar los datos por `area` y `subarea` para analizar cuáles son las disciplinas específicas más solicitadas en la plataforma de Konzerta.

In [ ]:
# Agrupamos por área y subárea para contar ofertas y sumar vacantes
tabla_subareas = (
    konzerta.groupby(["area", "subarea"], dropna=False)
    .agg(ofertas=("id", "size"), vacantes=("vacantes", "sum"))
    .reset_index()
    .sort_values(by="ofertas", ascending=False)
)

# Calculamos la participación porcentual de cada subárea sobre el total general
tabla_subareas["porcentaje_ofertas"] = (tabla_subareas["ofertas"] / len(konzerta) * 100).round(2)

# Mostramos las 15 subáreas con mayor número de ofertas
display(tabla_subareas.head(15))

,area,subarea,ofertas,vacantes,porcentaje_ofertas
40,"Comercial, Ventas y Negocios",Ventas,384,1077,12.75
34,"Comercial, Ventas y Negocios",Comercial,343,1625,11.39
12,"Administración, Contabilidad y Finanzas",Contabilidad,144,157,4.78
8,"Administración, Contabilidad y Finanzas",Administración,118,138,3.92
1,Abastecimiento y Logística,Almacén / Depósito / Expedición,96,202,3.19
30,"Atención al Cliente, Call Center y Telemarketing",Atención al Cliente,94,966,3.12
57,Gastronomía y Turismo,Gastronomia,78,203,2.59
5,Abastecimiento y Logística,Logística,77,145,2.56
113,Oficios y Otros,Otros,71,233,2.36
121,Producción y Manufactura,Mantenimiento,63,79,2.09


### **Visualización de las subáreas más demandadas**

Para facilitar la comprensión visual de las especialidades más buscadas, creamos un gráfico de barras horizontales con las 10 subáreas que concentran más ofertas en el dataset.

In [ ]:
# Seleccionamos las diez subáreas líderes
top_subareas = tabla_subareas.head(10).sort_values(by="ofertas")

fig_sub = px.bar(
    top_subareas,
    x="ofertas",
    y="subarea",
    orientation="h",
    text="ofertas",
    title="Las 10 subáreas profesionales con más ofertas en Konzerta",
    labels={"ofertas": "Número de ofertas (anuncios)", "subarea": "Subárea o especialidad"},
    color="area",
    color_discrete_sequence=px.colors.qualitative.Safe
)

fig_sub.update_layout(height=600, legend_title_text="Área Principal")
fig_sub.show()

### **Pregunta 2. Relación entre salarios y nivel de experiencia / jerarquía**

¿Cómo varían los salarios ofrecidos según el nivel de experiencia requerido en las ofertas que cuentan con información salarial?

Para responder esto de manera confiable, filtraremos los anuncios que tienen estado `'Registro sin alerta'` en la variable de control salarial. Agruparemos por nivel laboral para calcular estadísticas clave de los salarios máximos ofrecidos.

In [ ]:
# Filtramos ofertas con salarios válidos (sin alertas)
salarios_validos = konzerta[konzerta["estado_salario"] == "Registro sin alerta"]

# Agrupamos por nivel laboral para calcular el promedio, mediana y conteo de ofertas
resumen_salarios_nivel = (
    salarios_validos.groupby("nivel_laboral", dropna=False)
    .agg(
        ofertas=("id", "size"),
        salario_min_promedio=("salario_min", "mean"),
        salario_max_promedio=("salario_max", "mean"),
        salario_max_mediana=("salario_max", "median")
    )
    .reset_index()
    .sort_values(by="salario_max_mediana", ascending=False)
)

display(resumen_salarios_nivel.round(2))

,nivel_laboral,ofertas,salario_min_promedio,salario_max_promedio,salario_max_mediana
0,Jefe / Supervisor / Responsable,1,1300.00,1450.00,1450.0
5,Senior / Semi-Senior,10,NaN,1155.00,1050.0
1,Junior,15,888.89,1036.73,1000.0
3,Semi Sr,200,811.14,1051.12,989.0
2,Otro,4,675.00,912.50,900.0
4,Senior,2,787.50,900.00,900.0
7,Sin dato,181,794.43,1024.41,900.0
6,Sin Experiencia,1,550.00,750.00,750.0


### **Visualización de la distribución de salarios por Nivel Laboral**

Utilizaremos un gráfico de caja (box plot) interactivo para visualizar no solo los promedios, sino también la dispersión, mínimos, máximos y cuartiles de los salarios máximos ofrecidos según la jerarquía o nivel laboral solicitado.

In [ ]:
# Gráfico de cajas interactivo para los salarios según nivel laboral
fig_salarios = px.box(
    salarios_validos,
    x="nivel_laboral",
    y="salario_max",
    color="nivel_laboral",
    title="Distribución de salarios máximos ofrecidos por Nivel Laboral (Registros sin alerta)",
    labels={"nivel_laboral": "Nivel Laboral o Jerarquía", "salario_max": "Salario Máximo Ofrecido"},
    color_discrete_sequence=px.colors.qualitative.Safe
)

fig_salarios.update_layout(height=600, showlegend=False)
fig_salarios.show()

### **Pregunta 3. Modalidades de trabajo por área profesional**

¿Qué modalidades de trabajo son más frecuentes y cómo cambian según el área?

Primero, analizaremos la distribución general de la variable `modalidad` para observar cuál es la opción predominante en el mercado laboral capturado por Konzerta.

In [ ]:
# Distribución general de modalidades de trabajo
dist_modalidad = (
    konzerta["modalidad"].value_counts()
    .reset_index()
    .rename(columns={"count": "ofertas"})
)
dist_modalidad["porcentaje"] = (dist_modalidad["ofertas"] / len(konzerta) * 100).round(2)

display(dist_modalidad)

,modalidad,ofertas,porcentaje
0,Presencial,2942,97.71
1,Híbrido,59,1.96
2,Remoto,10,0.33


### **Relación entre la Modalidad y las 10 Áreas Principales**

Para comprender si la modalidad varía según la disciplina o tipo de profesión, cruzaremos estas variables enfocándonos en las 10 áreas con más anuncios.

In [ ]:
# Filtramos las ofertas pertenecientes a las 10 áreas principales
top_10_areas_nombres = top_areas["area"].tolist()
konzerta_top_areas = konzerta[konzerta["area"].isin(top_10_areas_nombres)]

# Tabla de contingencia cruzando area y modalidad
tabla_cruzada = (
    pd.crosstab(konzerta_top_areas["area"], konzerta_top_areas["modalidad"], normalize="index") * 100
).round(2).reset_index()

# Convertimos a formato largo para graficar más cómodamente
tabla_cruzada_larga = tabla_cruzada.melt(
    id_vars="area",
    value_vars=tabla_cruzada.columns[1:],
    var_name="modalidad",
    value_name="porcentaje"
)

# Gráfico de barras apiladas al 100%
fig_mod_area = px.bar(
    tabla_cruzada_larga,
    x="porcentaje",
    y="area",
    color="modalidad",
    orientation="h",
    title="Distribución porcentual de Modalidades de Trabajo en las 10 Áreas Principales",
    labels={"porcentaje": "Porcentaje (%)", "area": "Área Profesional", "modalidad": "Modalidad"},
    color_discrete_sequence=px.colors.qualitative.Pastel
)

fig_mod_area.update_layout(height=600, barmode="stack")
fig_mod_area.show()

### **Pregunta 4. Distribución Geográfica de las Oportunidades Laborales**

¿Cómo se distribuyen las ofertas según su ubicación geográfica (provincias y localidades)?

Analizaremos la participación porcentual de cada provincia para mapear dónde se concentran las ofertas de empleo publicadas en Konzerta.

In [ ]:
# Agrupamos y contamos ofertas por provincia
dist_provincia = (
    konzerta["provincia"].value_counts()
    .reset_index()
    .rename(columns={"count": "ofertas"})
)
dist_provincia["porcentaje"] = (dist_provincia["ofertas"] / len(konzerta) * 100).round(2)

display(dist_provincia)

,provincia,ofertas,porcentaje
0,Panamá,2358,78.31
1,Colón,148,4.92
2,Chiriquí,142,4.72
3,Panamá Oeste,121,4.02
4,Coclé,67,2.23
5,Veraguas,55,1.83
6,Herrera,47,1.56
7,Otra,42,1.39
8,Bocas del Toro,18,0.6
9,Los Santos,11,0.37


### **Visualización de la distribución por Provincia**

Creamos un gráfico interactivo para representar cómo se reparten las oportunidades de empleo en el territorio nacional.

In [ ]:
fig_prov = px.bar(
    dist_provincia,
    x="porcentaje",
    y="provincia",
    orientation="h",
    text="porcentaje",
    title="Distribución porcentual de Ofertas de Empleo por Provincia en Panamá",
    labels={"porcentaje": "Porcentaje del total (%)", "provincia": "Provincia"},
    color="provincia",
    color_discrete_sequence=px.colors.qualitative.Prism
)

fig_prov.update_traces(texttemplate='%{text:.2f}%', textposition='outside')
fig_prov.update_layout(height=500, showlegend=False)
fig_prov.show()

### **Pregunta 5. Evolución del volumen de ofertas por semana**

¿Cómo evoluciona el volumen de ofertas publicadas por semana entre julio y octubre de 2026?

Crearemos una columna temporal que represente la semana de publicación (usando la fecha de inicio de la semana) para agrupar las ofertas y analizar la tendencia a lo largo del tiempo.

In [ ]:
# Aseguramos que la fecha de publicación esté en formato datetime
konzerta["fecha_publicacion"] = pd.to_datetime(konzerta["fecha_publicacion"])

# Creamos una columna que representa el inicio de la semana de publicación (Lunes)
konzerta["semana_inicio"] = konzerta["fecha_publicacion"].dt.to_period("W").dt.start_time

# Agrupamos y contamos ofertas por semana
tendencia_semanal = (
    konzerta.groupby("semana_inicio")
    .size()
    .reset_index(name="ofertas")
    .sort_values(by="semana_inicio")
)

# Mostramos la tabla con los resultados
display(tendencia_semanal)

,semana_inicio,ofertas
0,2026-07-06,2
1,2026-07-13,40
2,2026-07-20,32
3,2026-07-27,30
4,2026-08-03,35
5,2026-08-10,275
6,2026-08-17,296
7,2026-08-24,281
8,2026-08-31,344
9,2026-09-07,416


### **Visualización de la tendencia de publicaciones**

Representamos la evolución semanal mediante un gráfico de línea para identificar picos de publicación y el comportamiento del mercado en el periodo de estudio.

In [ ]:
fig_linea = px.line(
    tendencia_semanal,
    x="semana_inicio",
    y="ofertas",
    title="Evolución semanal de ofertas de empleo publicadas (Julio - Octubre 2026)",
    labels={"semana_inicio": "Semana (Fecha de inicio)", "ofertas": "Cantidad de ofertas"},
    markers=True
)

fig_linea.update_traces(line_color="#1f77b4", line_width=3)
fig_linea.update_layout(height=500)
fig_linea.show()